# Benchmark on AMD

Measures throughput (img/s at batch 1, 8 and 32), p50/p95 latency and, when a price is given, cost per 1000
images for the trained UNet and the patch classifier, with `bench/bench.py`. The same script then runs on the
notebook's CPU for comparison. Every file in `results/bench-*.json` records the device name, torch and ROCm
versions, `amd-smi` output, the git commit and the exact command; the README table and the app's Benchmarks page
are generated from those files.

Cost: set `HOLDPOINT_PRICE_PER_HOUR` (USD per GPU hour) **and** `HOLDPOINT_PRICE_SOURCE` (where the price was read,
with the date). Without both, the cost fields stay empty.

In [ ]:
import os, sys, json
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(REPO)
sys.path.insert(0, str(REPO))

import torch
from holdpoint.vision import common
from bench import bench
DEV = common.pick_device("auto")
print(json.dumps(common.device_info(DEV), indent=2))
smi = bench.amd_smi()
print(smi if isinstance(smi, str) else json.dumps(smi.get("static"), indent=1)[:1500])

In [ ]:
RUNS = REPO / "runs"
SEG = Path(os.environ.get("HOLDPOINT_SEG_CKPT") or sorted(RUNS.glob("seg-*/model.pt"))[-1])
CLS = Path(os.environ.get("HOLDPOINT_CLS_CKPT") or sorted(RUNS.glob("cls-*/model.pt"))[-1])
IMAGES = os.environ.get("HOLDPOINT_BENCH_IMAGES", "data/raw/ozgenel_seg/extracted/rgb")
PRICE = []
if os.environ.get("HOLDPOINT_PRICE_PER_HOUR"):
    PRICE = ["--price-per-hour", os.environ["HOLDPOINT_PRICE_PER_HOUR"],
             "--price-source", os.environ["HOLDPOINT_PRICE_SOURCE"]]
print(SEG, common.sha256_file(SEG)[:12]); print(CLS, common.sha256_file(CLS)[:12]); print(IMAGES, PRICE)

In [ ]:
# Accelerator: UNet at 512 px and the classifier at its training size, 32 images cycled into each batch
gpu = []
for backend, ck in (("unet", SEG), ("classifier", CLS)):
    for amp in ([], ["--amp"]):
        r = bench.main(["--backend", backend, "--checkpoint", str(ck), "--images", IMAGES, "--limit", "32",
                        "--device", "auto", "--warmup", "3", "--iters", "20", *amp, *PRICE])
        gpu.append(r["path"])

In [ ]:
# CPU comparison on the same machine, same checkpoints and images (fewer iterations, no price)
cpu = []
for backend, ck in (("unet", SEG), ("classifier", CLS)):
    r = bench.main(["--backend", backend, "--checkpoint", str(ck), "--images", IMAGES, "--limit", "32",
                    "--device", "cpu", "--warmup", "1", "--iters", "3"])
    cpu.append(r["path"])

In [ ]:
# Summary and README table
for p in gpu + cpu:
    d = json.loads(Path(p).read_text())
    for row in d["rows"]:
        print(Path(p).name, d["backend"], d["settings"]["precision"], row["batch"], row["img_per_s"], "img/s",
              "p95", row["p95_ms"], "ms", row.get("usd_per_1000_images", ""))
from tools import results_table
results_table.main(["--write"])